# Environment Setup
Install required libraries for Kaggle T4x2 Environment.

In [ ]:
!pip install transformers mlflow scikit-learn easyocr

# 1. Imports and Reproducibility

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from tqdm import tqdm
import random
import numpy as np
import mlflow
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Dual-Stream NER Architecture

In [ ]:
class DualStreamFusionNER(nn.Module):
    def __init__(self, bert_hidden_size=768, num_classes=13, bert_model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super(DualStreamFusionNER, self).__init__()
        
        self.bert = AutoModel.from_pretrained(bert_model_name)
        
        self.vision_projection = nn.Linear(5, 128)
        self.vision_activation = nn.ReLU()
        
        self.fusion_projection = nn.Linear(bert_hidden_size + 128, 256)
        self.nlp_skip = nn.Linear(bert_hidden_size, 256)
        self.fusion_activation = nn.ReLU()
        
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(256, num_classes)
        
    def forward(self, input_ids, attention_mask, vision_meta):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        nlp_out = bert_outputs.last_hidden_state
        
        vis_out = self.vision_projection(vision_meta)
        vis_out = self.vision_activation(vis_out)
        
        batch_size, seq_len, _ = nlp_out.size()
        vis_out_expanded = vis_out.unsqueeze(1).expand(batch_size, seq_len, -1)
        
        fused = torch.cat((nlp_out, vis_out_expanded), dim=2)
        fused = self.fusion_projection(fused)
        
        nlp_residual = self.nlp_skip(nlp_out)
        fused = fused + nlp_residual
        
        fused = self.fusion_activation(fused)
        fused = self.dropout(fused)
        
        logits = self.classifier(fused)
        return logits

# 3. Dataset Configuration

In [ ]:
class FusedMedicalDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=512):
        self.records = []
        
        self.unique_tags = ['O', 'B-PATIENT_AGE', 'B-SIGNATURE', 'I-PATIENT_NAME', 'I-CLINIC_NAME', 
                            'B-CLINIC_NAME', 'B-DATE', 'I-MEDICATIONS', 'B-PATIENT_NAME', 
                            'B-CLINIC_ADDRESS', 'B-MEDICATIONS', 'I-CLINIC_ADDRESS', 'I-SIGNATURE']
        self.tag2id = {tag: i for i, tag in enumerate(self.unique_tags)}
        self.tag_counts = {tag: 0 for tag in self.unique_tags}
        
        with open(jsonl_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                if 'tokens' in data and 'iob_tags' in data:
                    self.records.append(data)
                    for tag in data['iob_tags']:
                        if tag in self.tag_counts:
                            self.tag_counts[tag] += 1
                            
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        record = self.records[idx]
        tokens = record['tokens']
        tags = record['iob_tags']
        
        encoding = self.tokenizer(
            tokens, 
            is_split_into_words=True, 
            return_offsets_mapping=False, 
            padding='max_length', 
            truncation=True, 
            max_length=self.max_length
        )
        
        labels = []
        word_ids = encoding.word_ids()
        
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                labels.append(-100)
            elif word_idx != previous_word_idx:
                tag = tags[word_idx] if word_idx < len(tags) else 'O'
                labels.append(self.tag2id.get(tag, self.tag2id['O']))
            else:
                labels.append(-100)
            previous_word_idx = word_idx
            
        input_ids = torch.tensor(encoding['input_ids'], dtype=torch.long)
        attention_mask = torch.tensor(encoding['attention_mask'], dtype=torch.long)
        labels_tensor = torch.tensor(labels, dtype=torch.long)
        
        avg_conf = record.get('avg_ocr_confidence', 0.0)
        ocr_features = record.get('ocr_features', [])
        if ocr_features:
            x1 = sum([f['bbox'][0][0] for f in ocr_features]) / len(ocr_features)
            y1 = sum([f['bbox'][0][1] for f in ocr_features]) / len(ocr_features)
            x2 = sum([f['bbox'][2][0] for f in ocr_features]) / len(ocr_features)
            y2 = sum([f['bbox'][2][1] for f in ocr_features]) / len(ocr_features)
        else:
            x1, y1, x2, y2 = 0.0, 0.0, 0.0, 0.0
            
        vision_meta = torch.tensor([avg_conf, x1, y1, x2, y2], dtype=torch.float32)
        
        return input_ids, attention_mask, vision_meta, labels_tensor

# 4. T4x2 Distributed Training Loop

In [ ]:
# KAGGLE DATASET PATH
data_path = "/kaggle/input/datasets/abhigtm19/medvision-nlp/unified_dataset_iob.jsonl"

if not os.path.exists(data_path):
    print(f"ERROR: Dataset not found at {data_path}. Please ensure you added the 'medvision-nlp' dataset to your notebook.")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    
    tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    full_dataset = FusedMedicalDataset(data_path, tokenizer, max_length=512)
    
    val_size = int(0.1 * len(full_dataset))
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    # Scale batch size for T4x2
    gpu_count = torch.cuda.device_count()
    base_batch_size = 16
    batch_size = base_batch_size * gpu_count if gpu_count > 0 else 8
    print(f"Found {gpu_count} GPUs. Scaling batch size to {batch_size}")
    
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    num_classes = len(full_dataset.unique_tags)
    model = DualStreamFusionNER(bert_hidden_size=768, num_classes=num_classes)
    
    # DataParallel for T4x2 Kaggle setup
    if gpu_count > 1:
        print(f"Wrapping model in nn.DataParallel to utilize all {gpu_count} GPUs...")
        model = nn.DataParallel(model)
    model.to(device)
    
    # Class weights
    counts = [full_dataset.tag_counts[tag] for tag in full_dataset.unique_tags]
    total_tokens = sum(counts)
    weights = [total_tokens / (num_classes * max(c, 1)) for c in counts]
    class_weights = torch.tensor(weights, dtype=torch.float).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
    optimizer = optim.AdamW(model.parameters(), lr=5e-5)
    
    best_val_f1 = 0.0
    patience = 5
    patience_counter = 0
    epochs = 10
    
    o_index = full_dataset.tag2id['O']
    
    # Optional MLFlow tracking (local to Kaggle)
    os.makedirs("/kaggle/working/tracking", exist_ok=True)
    mlflow.set_tracking_uri("sqlite:////kaggle/working/tracking/mlruns.db")
    mlflow.set_experiment("Kaggle_T4x2_NER")
    
    with mlflow.start_run():
        for epoch in range(epochs):
            # --- TRAINING ---
            model.train()
            train_loss = 0
            all_train_preds, all_train_labels = [], []
            
            progress = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]")
            for input_ids, attention_mask, vision_meta, labels in progress:
                input_ids = input_ids.to(device, non_blocking=True)
                attention_mask = attention_mask.to(device, non_blocking=True)
                vision_meta = vision_meta.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                optimizer.zero_grad(set_to_none=True)
                logits = model(input_ids, attention_mask, vision_meta)
                
                logits_flat = logits.view(-1, num_classes)
                labels_flat = labels.view(-1)
                loss = criterion(logits_flat, labels_flat)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                train_loss += loss.item()
                
                predictions = torch.argmax(logits_flat, dim=1)
                mask = labels_flat != -100
                all_train_preds.extend(predictions[mask].cpu().numpy())
                all_train_labels.extend(labels_flat[mask].cpu().numpy())
                
                progress.set_postfix({'loss': loss.item()})
                
            avg_train_loss = train_loss / len(train_dataloader)
            train_f1 = f1_score(all_train_labels, all_train_preds, labels=[i for i in range(num_classes) if i != o_index], average='macro', zero_division=0)
            
            # --- VALIDATION ---
            model.eval()
            val_loss = 0
            all_val_preds, all_val_labels = [], []
            
            with torch.no_grad():
                val_progress = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{epochs} [VAL]")
                for input_ids, attention_mask, vision_meta, labels in val_progress:
                    input_ids = input_ids.to(device, non_blocking=True)
                    attention_mask = attention_mask.to(device, non_blocking=True)
                    vision_meta = vision_meta.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    
                    logits = model(input_ids, attention_mask, vision_meta)
                    logits_flat = logits.view(-1, num_classes)
                    labels_flat = labels.view(-1)
                    loss = criterion(logits_flat, labels_flat)
                    
                    val_loss += loss.item()
                    
                    predictions = torch.argmax(logits_flat, dim=1)
                    mask = labels_flat != -100
                    all_val_preds.extend(predictions[mask].cpu().numpy())
                    all_val_labels.extend(labels_flat[mask].cpu().numpy())
                    
                    val_progress.set_postfix({'val_loss': loss.item()})
            
            avg_val_loss = val_loss / len(val_dataloader)
            val_f1 = f1_score(all_val_labels, all_val_preds, labels=[i for i in range(num_classes) if i != o_index], average='macro', zero_division=0)
            
            print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
                
                checkpoint_path = "/kaggle/working/dual_stream_ner_best.pth"
                checkpoint = {
                    'epoch': epoch,
                    'model_state_dict': model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
                    'val_f1': best_val_f1,
                    'tag2id': full_dataset.tag2id
                }
                torch.save(checkpoint, checkpoint_path)
                print(f"Saved best model at epoch {epoch+1} (Val F1: {best_val_f1:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch+1}! Overfitting detected.")
                    break
                    
    print("Training Complete! You can download '/kaggle/working/dual_stream_ner_best.pth'")